# CIFAR-10 Classification Hyperparameter Tuning

Contains the frozen-backprop sigma search, the documented local 3x3 grid, and an editable full-length run cell.

In [ ]:
from pathlib import Path
import os
import shutil
import sys


def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _looks_like_project_root(path: Path) -> bool:
    return (path / "learning_rules_MLP.py").is_file() and (path / "experiment_utils").is_dir()


def _find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for env_name in ["PROJECT_ROOT", "COLAB_PROJECT_ROOT"]:
        value = os.environ.get(env_name)
        if value:
            candidates.append(Path(value))
    candidates.extend([
        Path("/content/backprop-alternatives"),
        Path("/content/drive/MyDrive/backprop-alternatives"),
        Path("/content/drive/MyDrive/colab-folder"),
        Path("/content/drive/MyDrive/Colab Notebooks/drive-folder"),
    ])
    for start in candidates:
        try:
            resolved = start.expanduser().resolve()
        except Exception:
            continue
        for candidate in [resolved, *resolved.parents]:
            if _looks_like_project_root(candidate):
                return candidate
    if _running_in_colab():
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        for hit in Path("/content/drive/MyDrive").rglob("learning_rules_MLP.py"):
            candidate = hit.parent
            if _looks_like_project_root(candidate):
                return candidate
    raise FileNotFoundError("Could not find a folder containing learning_rules_MLP.py and experiment_utils/.")


def _stage_code_locally_if_colab(source_root: Path) -> Path:
    """Import code from /content in Colab instead of reading Python modules from Drive."""
    if not _running_in_colab() or not str(source_root).startswith("/content/drive/"):
        return source_root

    runtime_root = Path("/content/backprop-alternatives-runtime")
    if runtime_root.exists():
        shutil.rmtree(runtime_root)
    runtime_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_root / "learning_rules_MLP.py", runtime_root / "learning_rules_MLP.py")
    shutil.copytree(
        source_root / "experiment_utils",
        runtime_root / "experiment_utils",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc"),
    )
    (runtime_root / "notebooks").mkdir(exist_ok=True)
    return runtime_root


SOURCE_PROJECT_ROOT = _find_project_root()
PROJECT_ROOT = _stage_code_locally_if_colab(SOURCE_PROJECT_ROOT)
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from experiment_utils.runtime import get_device, setup_matplotlib

setup_matplotlib()
DEVICE = get_device()
print(f"Source project root: {SOURCE_PROJECT_ROOT}")
print(f"Runtime project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Device: {DEVICE}")


In [ ]:
NOTEBOOK_NAME = "cifar10-hyperparam-tuning"

TASK_CONFIG = {
    "task_key": "cifar10",
    "display_name": "CIFAR-10 classification",
    "task_type": "classification",
    "data_loader": "load_cifar10",
    "activation": "relu",
    "dimensions": (32 * 32 * 3, 512, 256, 10),
    "methods": ["bp", "np", "np_fixed", "np_fan_in", "wp"],
    "seeds": [0],
    "data_kwargs": {
        "train_limit": None,
        "test_limit": None,
        "train_eval_limit": 4000,
        "batch_size": 128,
        "eval_batch_size": 1024,
        "data_dir": str(DATA_DIR / "torchvision"),
        "seed": 0,
        "mean_center_only": True,
        "num_workers": 0,
    },
    "run_epochs": 80,
    "run_print_every_epoch": 10,
    "sigma_search": {
        "epochs": 50,
        "bp_lr": 0.100,
        "num_perturbations": 10,
        "batch_size": 128,
        "checkpoint_epochs": [1, 25, 50],
        "seeds": [0],
        "sigma_grids": {
            "np": [0.0145, 0.0160, 0.0175, 0.01875, 0.0200],
            "np_fan_in": [0.0045, 0.0050, 0.0055, 0.0060, 0.0065],
            "np_fixed": [0.140, 0.165, 0.190, 0.215, 0.240],
            "wp": [0.0095, 0.0110, 0.0125, 0.0140, 0.0155],
        },
    },
    "grid_search": {
        "epochs": 8,
        "seeds": [0],
        "print_every_epoch": 2,
        "local_grid": {
            "bp": {"lr": [0.075, 0.100, 0.125]},
            "np": {"lr": [0.0100, 0.0120, 0.0140], "sigma": [0.0160, 0.0175, 0.01875]},
            "np_fan_in": {"lr": [0.00550, 0.00667, 0.00800], "sigma": [0.0050, 0.0055, 0.0060]},
            "np_fixed": {"lr": [0.00320, 0.00400, 0.00480], "sigma": [0.165, 0.190, 0.215]},
            "wp": {"lr": [0.00120, 0.00160, 0.00200], "sigma": [0.0110, 0.0125, 0.0140]},
        },
    },
    "full_run_epochs": 80,
    "full_run_seeds": [0],
}

FULL_RUN_CONFIGS = {
    "bp": {"lr": 0.100},
    "np": {"lr": 0.0120, "sigma": 0.0175},
    "np_fan_in": {"lr": 0.00667, "sigma": 0.0055},
    "np_fixed": {"lr": 0.00400, "sigma": 0.190},
    "wp": {"lr": 0.00160, "sigma": 0.0125},
}


In [ ]:
from experiment_utils.sigma_search import run_sigma_search

sigma_outputs = run_sigma_search(TASK_CONFIG, project_root=PROJECT_ROOT, show=True, device=DEVICE)


In [ ]:
from experiment_utils.grid_search import best_grid_rows, run_local_grid_search
from IPython.display import display

grid_outputs = run_local_grid_search(TASK_CONFIG, project_root=PROJECT_ROOT, show=True, device=DEVICE)
print("Best row per method")
display(best_grid_rows(grid_outputs["grid_summary_df"]))


In [ ]:
# Editable full-length run. Change FULL_RUN_CONFIGS above, then set RUN_FULL_LENGTH=True.
RUN_FULL_LENGTH = False

if RUN_FULL_LENGTH:
    from experiment_utils.grid_search import run_full_length_training

    full_outputs = run_full_length_training(
        TASK_CONFIG,
        run_configs=FULL_RUN_CONFIGS,
        project_root=PROJECT_ROOT,
        show=True,
        device=DEVICE,
    )


In [ ]:
# Optional manual export/download. Set EXPORT_RESULTS=True after the runs have completed.
EXPORT_RESULTS = False

if EXPORT_RESULTS:
    from experiment_utils.export import download_if_colab, export_outputs

    export_bundle = {}
    for prefix, obj_name in [
        ("sigma", "sigma_outputs"),
        ("grid", "grid_outputs"),
        ("full", "full_outputs"),
    ]:
        if obj_name in globals():
            for key, value in globals()[obj_name].items():
                export_bundle[f"{prefix}_{key}"] = value
    archive_path = export_outputs(export_bundle, archive_name=NOTEBOOK_NAME)
    download_if_colab(archive_path)
